In [1]:
%pip install python-dotenv requests

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../assignments/.env")
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***")

16EB***


Q1. 우리말샘에서 단어 검색하기

(a) search_word 함수

In [7]:
import requests

def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    url = "https://opendict.korean.go.kr/api/search"  # 우리말샘 검색 엔드포인트
    
    params = {
        "key": KEY,         # 사전 준비에서 로드한 API 키
        "q": q,             # 검색할 단어
        "req_type": "json", # 응답 형식을 JSON으로 지정
        "num": num,         # 가져올 결과 수
        "start": start,     # 검색 시작 위치
        "type1": "word"     # 검색 대상을 단어로 한정
    }
    
    r = requests.get(url, params=params, timeout=10)  # timeout=10으로 요청
    r.raise_for_status()  # HTTP 오류 발생 시 예외 처리
    return r.json()       # 응답을 dict로 변환하여 반환

설명: 
search_word 함수는 우리말샘 공개 API의 /api/search 엔드포인트에 GET 요청을 보내 단어 검색 결과를 반환한다. raise_for_status()로 HTTP 오류를 검사한 후 응답을 JSON 형태의 딕셔너리로 반환하여 기본적으로 최대 10개의 결과를 1번째 항목부터 가져온다.

(b) 응답 구조 살펴보기

In [8]:
import json

data = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])  # 한글 그대로 출력, 앞 400자만 확인

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


설명: 
json.dumps()로 dict를 보기 좋게 들여쓰기하여 출력한다. 만약 ensure_ascii=False를 뺀다면 한글이 \uae40\uce58와 같은 유니코드 이스케이프 시퀀스로 출력되어 읽기 어려워진다.

(c) 원하는 정보만 추출

In [9]:
total = data["channel"]["total"]                     # 전체 검색 결과 수
items = data["channel"]["item"]                      # 이 페이지에서 받은 항목 리스트
n = len(items)                                       # 이 페이지 항목 수

print(f"총 {total}건,  이 페이지 {n}건")

for item in items[:5]:                               # 첫 5개 항목만 순회
    word = item["word"]                              # 표제어
    pos = item.get("pos", "품사 없음")                # 품사 (필드 없으면 "품사 없음")
    definition = item["sense"][0]["definition"][:40] # 뜻풀이 앞 40자
    print(f"{word} ({pos}) -> {definition}")

총 328건,  이 페이지 10건
김치 (품사 없음) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) -> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


설명: 
data["channel"]에서 전체 결과 수와 항목 리스트를 추출하고, 각 항목의 표제어, 품사, 뜻풀이를 출력한다. pos 필드가 없는 항목도 있기에 dict.get()으로 안전하게 접근하여 없다면 "품사 없음"을 기본값으로 사용한다.

Q2. 여러 검색어로 비교하기

In [17]:
import time
import collections

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥",
]

results: list[dict] = []  # 각 단어의 검색 결과를 저장할 리스트

for q in words:
    data = search_word(q)       # Q1의 search_word 호출
    results.append(data)        # 결과 저장
    time.sleep(0.3)             # 서버 부담 방지

설명: 
8개의 음식 관련 검색어에 대해 search_word()를 반복 호출하여 결과를 results 리스트에 저장한다. 매 호출 사이에 time.sleep(0.3)을 넣어 서버에 과도한 요청이 가지 않도록 한다.

(a) 검색어별 결과 수 출력

In [11]:
for q, data in zip(words, results):
    total = data["channel"]["total"]    # 전체 결과 수
    print(f"{q}: {total}건")

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건


설명: 
zip()으로 검색어와 결과를 묶어 순회하며, 각 검색어에 대한 전체 결과 수를 data["channel"]["total"]에서 추출하여 한 줄씩 출력하게 하였다.

(b) 품사 빈도 상위 3개

In [12]:
all_items: list[dict] = []
for data in results:
    all_items.extend(data["channel"]["item"])   # 8개 결과의 항목을 하나의 리스트로 합침

pos_list: list[str] = [
    item.get("pos") or "(미상)"                 # pos 없거나 빈 문자열이면 "(미상)"
    for item in all_items
]

counter = collections.Counter(pos_list)         # 품사별 빈도 계산
for pos, count in counter.most_common(3):       # 상위 3개 출력
    print(f"{pos}: {count}개")

(미상): 70개


설명: 
8개 검색어의 모든 항목을 하나의 리스트로 합친 후 collections.Counter로 품사 빈도를 계산하고,  most_common(3)으로 상위 3개를 출력한다. pos 필드가 없거나 빈 문자열인 항목은 "(미상)"으로 처리한다.

관찰: 
가장 흔한 품사는 명사일 것이다. 음식 관련 단어들은 대부분 사물이나 개념을 지칭하는 명사로 등재되기 때문이다.